# Meta Ads — Bronze Ingest

Lands connector CSV extracts into `bronze.*` Delta tables (schema-on-read envelope + `raw_json`).

**Prereq:** upload CSVs to `Files/landing/meta/` in the attached lakehouse.

In [ ]:
landing_path = "Files/landing/meta"
bronze_schema = "bronze"
overwrite = True

In [ ]:
from pyspark.sql import functions as F

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")

sources = {
    "meta_campaigns": f"{landing_path}/meta_campaigns*.csv",
    "meta_adsets": f"{landing_path}/meta_adsets*.csv",
    "meta_ads": f"{landing_path}/meta_ads*.csv",
    "meta_adset_insights": f"{landing_path}/meta_adset_insights*.csv",
    "meta_ad_insights": f"{landing_path}/meta_ad_insights*.csv",
    "meta_path_verify": f"{landing_path}/meta_path_verify*.csv",
}

for table, path in sources.items():
    df = (
        spark.read.option("header", True).option("multiLine", True).option("escape", '"').csv(path)
        .withColumn("bronze_ingested_at", F.current_timestamp())
        .withColumn("source_file", F.input_file_name())
    )
    target = f"{bronze_schema}.{table}"
    mode = "overwrite" if overwrite else "append"
    (
        df.write.format("delta")
        .mode(mode)
        .option("overwriteSchema", "true")
        .saveAsTable(target)
    )
    print(f"{target}: {df.count():,} rows")